# Phase 5b: CAD with class-weighted loss + context-dependent analysis (~1 h)

**Settings:** Accelerator **GPU T4 x2**, **Internet on**. **Attach:** `phase5-cad` (required: its CAD pairs, LLM outputs and runs).

1. RoBERTa with a class-weighted loss (balanced: counter-speech, 2.5% of training pairs, counts about 13x) for base / +LLM /
   +LLM+label, 3 seeds. No LLM work is repeated.
2. Significance report over all CAD runs, now with a section on context-dependent vs other test pairs:
   `/kaggle/working/outputs/significance_weighted_f1.md` (per-class F1 for the new runs: `ablation_weighted_f1.md`).

In [ ]:
!nvidia-smi -L
import transformers, torch
print("torch", torch.__version__, "| transformers", transformers.__version__)
assert torch.cuda.is_available(), "NO GPU! Settings -> Accelerator -> GPU T4 x2, then run again. (On CPU every run takes hours.)"


In [ ]:
# Clone (or update) the project code and the dataset.
import os, subprocess, glob
REPO = "akashnegetive/stance-aware-llm-explanation-augmented-hate-speech-detection-for-replies-in-conversation"
CODE_DIR = "/kaggle/working/code"
if os.path.isdir(CODE_DIR):
    subprocess.run(["git", "-C", CODE_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{REPO}.git", CODE_DIR], check=True)
%cd {CODE_DIR}
if not os.path.isdir("data/counter_context"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/xinchenyu/counter_context", "data/counter_context"], check=True)

In [ ]:
import shlex
prev = sorted(glob.glob("/kaggle/input/**/outputs", recursive=True))
cad = sorted(glob.glob("/kaggle/input/**/cad_data/gold", recursive=True))
llm = sorted(glob.glob("/kaggle/input/**/llm_cad/llm_outputs.jsonl", recursive=True))
labels = sorted(glob.glob("/kaggle/input/**/llm_cad_Qwen2.5-7B-Instruct_2shot/raw.jsonl", recursive=True))
assert prev and cad and llm and labels, "Attach the phase5-cad notebook as an input first"
CAD = "/kaggle/working/cad_data"
subprocess.run(["cp", "-r", os.path.dirname(cad[0]), CAD], check=True)
LLM_FILE, LABEL_FILE = llm[0], labels[0]
DIRS = " ".join(shlex.quote(d) for d in prev + ["/kaggle/working/outputs"])
print("CAD pairs:", CAD, "| LLM file:", LLM_FILE, "| labels:", LABEL_FILE)
print("earlier runs:", prev)

In [ ]:
# 1. Class-weighted RoBERTa on CAD (one GPU). The class weights are printed at the start of each run.
!CUDA_VISIBLE_DEVICES=0 python ablation.py --dataset cad --data_dir {CAD} --class_weights balanced --models roberta-base=1e-5 --variants base llm llm_label --seeds 13 42 87 --select_metric weighted_f1 --llm_file {LLM_FILE} --llm_label_file {LABEL_FILE} --output_dir /kaggle/working/outputs --prev_results {DIRS}

In [ ]:
# 2. Significance report over all CAD runs (unweighted from phase5-cad + class-weighted), incl. context-dependent pairs.
!python significance.py --results_dirs {DIRS} --data_dir {CAD} --llm_file {LLM_FILE} --out_dir /kaggle/working/outputs